# Climate Engine API to SPI

Pull a precipitation time series and a standardized index for one point from the
Climate Engine API, then recompute the index here from the same raw series.

The API key goes in the `CE_API_KEY` environment variable. It is sent in the
`Authorization` header and is never written into this notebook or its outputs.
Keys are issued by the Climate Engine team and expire after about 60 days.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from climate_indices import spi
from climate_indices.compute import Periodicity
from climate_indices.drought.indices import Distribution

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from scripts.earthengine_climate_engine_pilot import (
    climate_engine_native,
    climate_engine_series,
    parse_climate_engine_series,
)

key = os.environ.get("CE_API_KEY", "").strip()
assert key, "set CE_API_KEY to a Climate Engine API key"
print("key loaded, length", len(key))

## Ask the API for its own SPI-3

`distribution=nonparametric` is the option that matches a rank-based
standardization. `accumulation` is expressed in the dataset's own unit, which is
days for the daily gridMET dataset, so a 3-month index is 90 days.

In [ ]:
CALIBRATION = (1981, 2016)
WINDOW = ("1981-01-01", "2024-12-31")

ce_index = climate_engine_series(key, 3, "spi")
ce_spi = parse_climate_engine_series(ce_index["response"], "spi")
print("request sent:", {k: v for k, v in ce_index["request"].items() if k != "coordinates"})
print("rows:", len(ce_spi), "from", ce_spi.index.min().date(), "to", ce_spi.index.max().date())
ce_spi.tail()

## Ask the API for the raw precipitation

The native endpoint returns the dataset's own values, in millimetres per day for
gridMET. Summing each calendar month gives the input the NumPy API expects.

In [ ]:
ce_native = climate_engine_native(key, "GRIDMET", "pr", *WINDOW)
daily_precipitation = parse_climate_engine_series(ce_native["response"], "pr")
print("daily rows:", len(daily_precipitation))

monthly_precipitation = daily_precipitation.resample("MS").sum()
monthly_precipitation = monthly_precipitation.where(monthly_precipitation.index <= pd.Timestamp(WINDOW[1]))
print("monthly mean: %.1f mm" % monthly_precipitation.mean())
assert 1.0 < monthly_precipitation.mean() < 500.0, "monthly total is not in millimetres; check the API unit"
monthly_precipitation.to_frame().tail()

## Compute SPI-3 here from the same precipitation

The API fits a rank-based distribution over its own sample; this call fits a
gamma distribution over 1981-2016. The two series therefore describe the same
precipitation with different methods, and a difference between them is expected.
What this shows is the pipeline: an API series becomes a `climate_indices`
input without a conversion step in between.

In [ ]:
ours = spi(
    monthly_precipitation.to_numpy(dtype="float64"),
    3,
    Distribution.gamma,
    int(monthly_precipitation.index[0].year),
    CALIBRATION[0],
    CALIBRATION[1],
    Periodicity.monthly,
)
ours_series = pd.Series(ours, index=monthly_precipitation.index, name="spi_gamma_here")

paired = pd.concat([ce_spi.rename("spi_api_nonparametric"), ours_series], axis=1, join="inner").dropna()
correlation = paired.corr().iloc[0, 1]
print(f"paired rows: {len(paired)}   correlation: {correlation:.4f}")
print(f"mean difference (ours - api): {(paired.iloc[:, 1] - paired.iloc[:, 0]).mean():+.4f}")
paired.tail()

In [ ]:
figure, axis = plt.subplots(figsize=(11, 4.5))
axis.plot(paired.index, paired["spi_api_nonparametric"], label="Climate Engine, nonparametric", linewidth=1.0)
axis.plot(paired.index, paired["spi_gamma_here"], label="climate_indices, gamma", linewidth=1.0, alpha=0.8)
axis.axhline(0.0, color="black", linewidth=0.8)
axis.set_ylabel("SPI-3")
axis.set_title("SPI-3 at one point, two methods and one precipitation series")
axis.legend(loc="lower left")
figure.tight_layout()

## What to watch for

- **Missing values arrive as `-9999`.** Left as a number, one sentinel would
  dominate a fit. `parse_climate_engine_series` turns it into NaN.
- **The calibration window is a parameter of the request**, not a property of
  the data. Two callers who ask for different windows get different indices from
  the same precipitation.
- **The distribution matters more than the pipeline.** These two series agree
  closely in shape and differ in the tails, because gamma and a rank-based fit
  do not have the same limits.